# 2단계: Sentence-BERT 리뷰 임베딩

1단계에서 정제한 리뷰(`data/processed/reviews.parquet`)를 Sentence-BERT로 임베딩한다.

## 이 노트북은 Colab(GPU)에서 실행하는 걸 전제로 작성했다

로컬 CPU로 속도 테스트를 해봤더니 전체 225,283건 임베딩에 2.5~5시간이 걸려서(내 컴퓨터엔 GPU가 없음), Colab GPU에서 돌리기로 했다. 로컬에서는 소규모 샘플(100~2000건)로 코드/의미 검증만 마쳤고, 전체 실행은 여기서 한다.

**Colab 실행 전 체크리스트** (코드는 이미 다 세팅되어 있음, 아래 두 개만 하면 됨)
1. 상단 메뉴 `런타임 > 런타임 유형 변경` → 하드웨어 가속기를 **GPU**로 설정
2. 왼쪽 파일 탭에 `data/processed/reviews.parquet`을 업로드 (파일명은 그대로 `reviews.parquet`)
3. 위 두 개만 하고 나면 셀을 처음부터 끝까지 순서대로 실행하면 됨. 마지막 셀에서 `embeddings.npy`, `embedding_ids.parquet`가 자동으로 다운로드된다 → 그걸 로컬 `data/processed/`에 넣으면 끝

In [ ]:
!pip install -q sentence-transformers pyarrow

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# 왼쪽 파일 탭에 reviews.parquet을 업로드한 뒤, 업로드한 파일명 그대로 지정
DATA_PATH = "reviews.parquet"
OUT_DIR = "."   # Colab 임시 저장소(/content/) -> 세션 끊기면 사라지므로 실행 끝나면 바로 다운로드할 것

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "GPU가 안 잡혔다. 런타임 > 런타임 유형 변경에서 GPU를 선택했는지 확인할 것"

## 1. 데이터 로드 + 모델 로드

`jhgan/ko-sroberta-multitask`: 한국어로 파인튜닝된 Sentence-BERT. 768차원 벡터를 출력한다. 로컬에서 이미 모델 로딩/추론 코드가 정상 동작하는 걸 확인했다 (`docs/personal/04_2단계_코드_기술_설명.md` 참고).

In [ ]:
df = pd.read_parquet(DATA_PATH)
print("리뷰 수:", len(df))
assert df["RawText_clean"].isnull().sum() == 0

t0 = time.time()
model = SentenceTransformer("jhgan/ko-sroberta-multitask", device=device)
print(f"모델 로딩: {time.time()-t0:.1f}초, 임베딩 차원: {model.get_sentence_embedding_dimension()}")

## 2. 코드 정합성 검증 (본 실행 전 소규모 상식 체크)

전체 22만 건을 다 돌리기 전에, 결과가 상식적으로 말이 되는지 작은 샘플로 먼저 확인한다. 로컬 CPU 테스트에서 이미 통과한 체크지만 Colab 환경(다른 GPU/torch 버전)에서도 다시 한번 확인.

- 완전히 같은 문장 → 코사인 유사도 1.0
- 의미 비슷한 문장(둘 다 배터리 불만) → 유사도 높게
- 의미 다른 문장(배터리 vs 디자인) → 유사도 낮게

In [ ]:
def cos_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sanity_pairs = [
    ("배터리가 빨리 닳아요", "충전이 오래 못 가네요", "유사(배터리 불만)"),
    ("배터리가 빨리 닳아요", "디자인이 정말 예쁘네요", "비유사(주제 다름)"),
    ("사이즈가 딱 맞아요", "사이즈가 딱 맞아요", "완전동일"),
]
flat = [t for a, b, _ in sanity_pairs for t in (a, b)]
e = model.encode(flat, show_progress_bar=False)
for i, (a, b, desc) in enumerate(sanity_pairs):
    s = cos_sim(e[2*i], e[2*i+1])
    print(f"[{desc}] cos_sim={s:.3f}")
    if desc == "완전동일":
        assert abs(s - 1.0) < 1e-4, "동일 문장인데 유사도가 1이 아님 -> 코드 점검 필요"

print("\n상식 검증 통과")

## 3. 전체 리뷰 임베딩 (GPU)

`RawText_clean`(1단계 정제 텍스트) 기준으로 임베딩한다. GPU면 몇 분 안에 끝날 것으로 예상.

In [ ]:
texts = df["RawText_clean"].tolist()

t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"\n전체 임베딩 완료: {elapsed/60:.1f}분, shape={embeddings.shape}, dtype={embeddings.dtype}")

## 4. 전체 결과 검증

작게 확인한 상식 검증을 전체 결과에 대해서도 확인한다.

1. shape이 (리뷰 수, 768)인지, NaN/Inf가 없는지
2. 벡터 norm이 극단적으로 튀는(0에 가깝거나 비정상적으로 큰) 행이 있는지
3. 1단계 EDA에서 찾은 "완전히 같은 텍스트인데 도메인만 다른" 실제 쌍들의 유사도가 1.0에 가까운지 (이미 전처리 단계에서 검증된 사실을 임베딩에서도 재확인)
4. 무작위로 뽑은 리뷰 하나와 가장 가까운 이웃(nearest neighbor)이 실제로 의미상 비슷한지 육안 확인

In [ ]:
# 1) shape / NaN / Inf
assert embeddings.shape == (len(df), 768), "shape이 예상과 다름"
assert not np.isnan(embeddings).any(), "NaN 존재"
assert not np.isinf(embeddings).any(), "Inf 존재"
print("shape/NaN/Inf 검증 통과:", embeddings.shape)

# 2) norm 분포
norms = np.linalg.norm(embeddings, axis=1)
print(f"norm: min={norms.min():.3f}, max={norms.max():.3f}, mean={norms.mean():.3f}, std={norms.std():.3f}")
print("norm이 0에 가까운(사실상 빈 벡터) 리뷰 수:", (norms < 1e-3).sum())

# 3) 도메인 교차 완전동일 텍스트 쌍 재검증
dup_groups = list(df[df.duplicated(subset=["RawText_clean"], keep=False)].groupby("RawText_clean"))
print(f"\n도메인 교차 완전동일 텍스트 그룹: {len(dup_groups)}개")
sims = []
for text, g in dup_groups[:10]:
    idxs = g.index[:2]
    a, b = embeddings[df.index.get_loc(idxs[0])], embeddings[df.index.get_loc(idxs[1])]
    sims.append(cos_sim(a, b))
print(f"완전동일 텍스트 쌍 유사도 평균: {np.mean(sims):.4f} (1.0에 가까워야 정상)")

# 4) 최근접 이웃 육안 확인
rng = np.random.default_rng(42)
query_idx = int(rng.integers(0, len(df)))
sims_to_query = embeddings @ embeddings[query_idx] / (norms * norms[query_idx])
top5 = np.argsort(-sims_to_query)[:6]  # 0번은 자기 자신
print(f"\n쿼리 리뷰: {df.iloc[query_idx]['RawText_clean'][:60]}")
print("가장 가까운 이웃 5개:")
for i in top5[1:]:
    print(f"  sim={sims_to_query[i]:.3f} | {df.iloc[i]['RawText_clean'][:60]}")

## 5. (덤) 유사 중복(fuzzy duplicate) 탐색

1단계에서 완전히 똑같은 텍스트만 중복 제거했고, "살짝 다른데 사실상 같은 리뷰"는 못 걸렀었다. 임베딩이 생겼으니 유사도 0.98 이상인데 텍스트가 100% 동일하지는 않은 쌍이 있는지 가볍게 훑어본다 (전수조사는 22만×22만이라 무거우니 무작위 5000건 샘플로만).

In [ ]:
sample_n = 5000
rng = np.random.default_rng(0)
idx_sample = rng.choice(len(df), size=min(sample_n, len(df)), replace=False)
E = embeddings[idx_sample]
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)
sim_matrix = E_norm @ E_norm.T
np.fill_diagonal(sim_matrix, -1)  # 자기 자신 제외

pairs_found = np.argwhere(sim_matrix >= 0.98)
pairs_found = pairs_found[pairs_found[:, 0] < pairs_found[:, 1]]  # 중복쌍 제거
print(f"샘플 {sample_n}건 중 유사도 0.98+ 쌍: {len(pairs_found)}개")

texts_sample = df.iloc[idx_sample]["RawText_clean"].tolist()
shown = 0
for i, j in pairs_found:
    a, b = texts_sample[i], texts_sample[j]
    if a != b:  # 완전동일(이미 1단계에서 처리된 도메인교차 케이스 등)은 제외하고 진짜 '거의 같은' 것만
        print(f"sim={sim_matrix[i,j]:.3f}")
        print("  A:", a[:70])
        print("  B:", b[:70])
        shown += 1
    if shown >= 10:
        break
if shown == 0:
    print("텍스트가 다르면서 유사도 0.98+인 쌍 없음 (샘플 내에서는 fuzzy duplicate 문제 미발견)")

## 6. 저장

`review_id` 순서와 임베딩 행 순서가 반드시 일치해야 하므로, 임베딩 배열과 `review_id` 리스트를 같이 저장한다. 다운로드해서 로컬 `data/processed/`에 넣으면 된다.

In [ ]:
import os

os.makedirs(OUT_DIR, exist_ok=True)
np.save(os.path.join(OUT_DIR, "embeddings.npy"), embeddings.astype(np.float32))
df[["review_id"]].to_parquet(os.path.join(OUT_DIR, "embedding_ids.parquet"), index=False)

print("저장 완료:")
print(" -", os.path.join(OUT_DIR, "embeddings.npy"), embeddings.shape)
print(" -", os.path.join(OUT_DIR, "embedding_ids.parquet"), len(df), "행 (review_id 순서 = embeddings.npy 행 순서)")

# 파일 업로드 방식이라 세션 끊기면 날아감 -> 끝나면 바로 다운로드
from google.colab import files
files.download(os.path.join(OUT_DIR, "embeddings.npy"))
files.download(os.path.join(OUT_DIR, "embedding_ids.parquet"))